# Day 9 — Single agent vs lead + subagents
**Theory:** [day-09-multi-agent-systems.md](../theory/day-09-multi-agent-systems.md) · ⏱ ~75 min · Level: medium

Task: compare ShopEase's return policy with 3 competitor policies and list the gaps.
This is **parallel, read-only research**, the kind of work where multi-agent can pay off. We'll measure.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
# Please don't let an AI "auto-fix" rewrite the imports below: the course packages are `course_utils` and `shopease`.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone (or update) the course repo, install it, read the key from notebook secrets
    BRANCH = os.getenv("COURSE_BRANCH", "main")
    REPO_URL = os.getenv("COURSE_REPO_URL", "https://github.com/Rahulverma5/agentic-ai-course")
    REPO = pathlib.Path(os.getenv("COURSE_BASE_DIR", "/content" if IN_COLAB else "/kaggle/working")) / "agentic-ai-course"
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:  # an older clone from a previous session: bring it up to date
        subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt"), "-e", str(REPO)],
                   check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets (🔑 icon in Colab, Add-ons → Secrets in Kaggle):", e)
else:  # Codespaces / local: use the checked-out repo
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
try:
    from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
    from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
    from shopease.data import setup_data
except ImportError as e:
    raise ImportError(
        f"{e}\n\nThe course code at {REPO} is missing or outdated. Fix it like this:\n"
        f"  • Colab/Kaggle: run  !rm -rf {REPO}  then Runtime → Restart session, and run this cell again.\n"
        "  • Codespaces/local: run  git pull && pip install -e .  in a terminal, then restart the kernel.") from None
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. The "documents" each researcher can read

In [ ]:
import time, concurrent.futures as cf
import pandas as pd
from course_utils import run_agent

DOCS = {
    "shopease": pathlib.Path("projects/shopease-support-agent/data/policies/returns.md").read_text(),
    "quickcart": "Returns within 60 days. Free return shipping for everyone. Opened electronics: 15-day return window, "
                 "restocking fee 10%. Refunds in 3 business days.",
    "megamart": "Returns within 14 days with receipt. Return shipping $7. No returns on opened electronics. "
                "Store credit only after 14 days, up to 30 days.",
    "trendly": "30-day returns. Free returns for members. Exchanges are free and processed first. Refund to original "
               "payment in 5-10 days. Defective items: 1-year no-questions replacement.",
}
READ = {"name": "read_policy", "description": "Read the full return policy of one store: shopease, quickcart, megamart or trendly.",
        "input_schema": {"type": "object", "properties": {"store": {"type": "string"}}, "required": ["store"]}}
read_policy = lambda store: DOCS.get(store.lower(), f"Error: unknown store {store}. Options: {', '.join(DOCS)}")
TASK = ("Compare ShopEase's return policy with quickcart, megamart and trendly. Output a table (window, shipping cost, "
        "electronics rule, refund speed) and the 3 most important gaps ShopEase should fix.")

## 2. (A) One agent does everything

In [ ]:
t0, c0 = time.time(), tracker.total()
single = run_agent(TASK, [READ], {"read_policy": read_policy}, thinking_level="medium", verbose=False)
single_stats = {"design": "single agent", "seconds": round(time.time() - t0, 1), "cost $": round(tracker.total() - c0, 4),
                "steps": single.steps}
print(single.text)

## 3. (B) Lead agent + 3 parallel subagents (cheap model, fresh context each)
Each subagent returns a **compressed** summary. The lead never sees the raw documents.

In [ ]:
SUB_PROMPT = ("You are a research subagent. Read ONLY the policy of store '{store}' using your tool, then return a "
              "≤60-word summary with: window, shipping cost, electronics rule, refund speed. No other text.")
def subagent(store):
    res = run_agent(SUB_PROMPT.format(store=store), [READ], {"read_policy": read_policy},
                    model=CHEAP_MODEL, max_steps=4, verbose=False)
    return store, res.text

t0, c0 = time.time(), tracker.total()
with cf.ThreadPoolExecutor() as pool:
    summaries = dict(pool.map(subagent, ["shopease", "quickcart", "megamart", "trendly"]))
for s, txt in summaries.items():
    print(f"[{s}] {txt}\n")
lead_input = "\n\n".join(f"## {s}\n{txt}" for s, txt in summaries.items())
lead = ask(f"Research summaries from subagents:\n{lead_input}\n\n{TASK}", model=MODEL, max_tokens=3000)
multi_stats = {"design": "lead + 3 subagents", "seconds": round(time.time() - t0, 1),
               "cost $": round(tracker.total() - c0, 4), "steps": "parallel"}
print(lead)

## 4. Compare
Score quality yourself (1–5) after reading both answers. Did the multi-agent version lose details in summarisation?

In [ ]:
pd.DataFrame([single_stats, multi_stats])

## 5. "Many readers, one writer"
Rule of thumb from today's theory: subagents **gather**, one agent **decides and writes**. Here, subagents only read
policies; the lead alone writes the recommendation. Never let two agents edit the same document or reply
to the same customer.

## ✏️ Exercises
1. Give the subagents vague instructions ("research this store"). What happens to the summaries and the final table?
2. Make subagents return structured output (a Pydantic model) instead of text. Does the lead's table get more accurate?
3. Estimate when (B) is worth it: if (B) is 10% better, how many $ of value per task justify its extra cost?

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None